In [2]:
!pip install pandas_ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 15.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstal

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import yfinance as yf
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_curve, precision_score, mean_squared_error, mean_absolute_error)
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

import pandas_ta as ta
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

tickers = [
"AAPL","MSFT","GOOG","AMZN","META","NVDA","ADBE","ORCL","CRM","INTC",
"JPM","BAC","WFC","V","MA","GS",
"WMT","COST","PG","KO","PEP","HD","MCD",
"UNH","JNJ","PFE",
"XOM","CVX",
"TSLA","BA","CAT",
"HDFCBANK.NS", "ICICIBANK.NS", "SBIN.NS", "AXISBANK.NS", "KOTAKBANK.NS",
"BAJFINANCE.NS", "BAJAJFINSV.NS", "HDFCLIFE.NS",
"TCS.NS", "INFY.NS", "HCLTECH.NS", "WIPRO.NS", "TECHM.NS",
"RELIANCE.NS", "ONGC.NS", "NTPC.NS", "POWERGRID.NS",
"HINDUNILVR.NS", "ITC.NS", "NESTLEIND.NS", "BRITANNIA.NS", "DABUR.NS",
"MARUTI.NS", "TMPV.NS", "M&M.NS", "BAJAJ-AUTO.NS",
"TATASTEEL.NS", "JSWSTEEL.NS", "LT.NS",
"SUNPHARMA.NS", "DRREDDY.NS"
]

TICKER_META = {
    # Tech
    "AAPL": {"market": "US", "cap_size": "large", "sector": "tech"},
    "MSFT": {"market": "US", "cap_size": "large", "sector": "tech"},
    "GOOG": {"market": "US", "cap_size": "large", "sector": "tech"},
    "AMZN": {"market": "US", "cap_size": "large", "sector": "ecommerce"},
    "META": {"market": "US", "cap_size": "large", "sector": "tech"},
    "NVDA": {"market": "US", "cap_size": "large", "sector": "semiconductors"},
    "ADBE": {"market": "US", "cap_size": "large", "sector": "software"},
    "ORCL": {"market": "US", "cap_size": "large", "sector": "software"},
    "CRM":  {"market": "US", "cap_size": "large", "sector": "software"},
    "INTC": {"market": "US", "cap_size": "large", "sector": "semiconductors"},

    # Finance
    "JPM": {"market": "US", "cap_size": "large", "sector": "banking"},
    "BAC": {"market": "US", "cap_size": "large", "sector": "banking"},
    "WFC": {"market": "US", "cap_size": "large", "sector": "banking"},
    "V":   {"market": "US", "cap_size": "large", "sector": "payments"},
    "MA":  {"market": "US", "cap_size": "large", "sector": "payments"},
    "GS":  {"market": "US", "cap_size": "large", "sector": "investment_banking"},

    # Consumer
    "WMT": {"market": "US", "cap_size": "large", "sector": "retail"},
    "COST":{"market": "US", "cap_size": "large", "sector": "retail"},
    "PG":  {"market": "US", "cap_size": "large", "sector": "consumer_goods"},
    "KO":  {"market": "US", "cap_size": "large", "sector": "beverages"},
    "PEP": {"market": "US", "cap_size": "large", "sector": "beverages"},
    "HD":  {"market": "US", "cap_size": "large", "sector": "home_improvement"},
    "MCD": {"market": "US", "cap_size": "large", "sector": "restaurants"},

    # Healthcare
    "UNH": {"market": "US", "cap_size": "large", "sector": "healthcare"},
    "JNJ": {"market": "US", "cap_size": "large", "sector": "pharma"},
    "PFE": {"market": "US", "cap_size": "large", "sector": "pharma"},

    # Energy
    "XOM": {"market": "US", "cap_size": "large", "sector": "oil_gas"},
    "CVX": {"market": "US", "cap_size": "large", "sector": "oil_gas"},

    # Others
    "TSLA":{"market": "US", "cap_size": "large", "sector": "ev"},
    "BA":  {"market": "US", "cap_size": "large", "sector": "aerospace"},
    "CAT": {"market": "US", "cap_size": "large", "sector": "machinery"},
    # Banking & Finance
    "HDFCBANK.NS": {"market": "India", "cap_size": "large", "sector": "banking"},
    "ICICIBANK.NS": {"market": "India", "cap_size": "large", "sector": "banking"},
    "SBIN.NS": {"market": "India", "cap_size": "large", "sector": "banking"},
    "AXISBANK.NS": {"market": "India", "cap_size": "large", "sector": "banking"},
    "KOTAKBANK.NS": {"market": "India", "cap_size": "large", "sector": "banking"},
    "BAJFINANCE.NS": {"market": "India", "cap_size": "large", "sector": "nbfc"},
    "BAJAJFINSV.NS": {"market": "India", "cap_size": "large", "sector": "financial_services"},
    "HDFCLIFE.NS": {"market": "India", "cap_size": "large", "sector": "insurance"},

    # IT / Tech
    "TCS.NS": {"market": "India", "cap_size": "large", "sector": "it_services"},
    "INFY.NS": {"market": "India", "cap_size": "large", "sector": "it_services"},
    "HCLTECH.NS": {"market": "India", "cap_size": "large", "sector": "it_services"},
    "WIPRO.NS": {"market": "India", "cap_size": "large", "sector": "it_services"},
    "TECHM.NS": {"market": "India", "cap_size": "large", "sector": "it_services"},

    # Energy / Infra
    "RELIANCE.NS": {"market": "India", "cap_size": "large", "sector": "oil_gas"},
    "ONGC.NS": {"market": "India", "cap_size": "large", "sector": "oil_gas"},
    "NTPC.NS": {"market": "India", "cap_size": "large", "sector": "power"},
    "POWERGRID.NS": {"market": "India", "cap_size": "large", "sector": "power"},

    # FMCG / Consumer
    "HINDUNILVR.NS": {"market": "India", "cap_size": "large", "sector": "fmcg"},
    "ITC.NS": {"market": "India", "cap_size": "large", "sector": "fmcg"},
    "NESTLEIND.NS": {"market": "India", "cap_size": "large", "sector": "fmcg"},
    "BRITANNIA.NS": {"market": "India", "cap_size": "large", "sector": "fmcg"},
    "DABUR.NS": {"market": "India", "cap_size": "large", "sector": "fmcg"},

    # Auto
    "MARUTI.NS": {"market": "India", "cap_size": "large", "sector": "auto"},
    "TMPV.NS": {"market": "India", "cap_size": "large", "sector": "auto"},
    "M&M.NS": {"market": "India", "cap_size": "large", "sector": "auto"},
    "BAJAJ-AUTO.NS": {"market": "India", "cap_size": "large", "sector": "auto"},

    # Metals / Industrials
    "TATASTEEL.NS": {"market": "India", "cap_size": "large", "sector": "metals"},
    "JSWSTEEL.NS": {"market": "India", "cap_size": "large", "sector": "metals"},
    "LT.NS": {"market": "India", "cap_size": "large", "sector": "infrastructure"},

    # Pharma
    "SUNPHARMA.NS": {"market": "India", "cap_size": "large", "sector": "pharma"},
    "DRREDDY.NS": {"market": "India", "cap_size": "large", "sector": "pharma"},
}

CONFIG = {
    "ffill_limit":       2,
    "return_horizon":    5,       # predict N-day forward return
    "target_threshold":  0.005,   # +0.5% = Class 1
    "outlier_sigma":     3,        # volume outlier z-score
    "vol_window":        252,      # rolling normalization window
    "confidence_thresh": 0.60,    # minimum confidence to trade
}

In [22]:
data = yf.download(
    tickers,
    period="2y",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    threads=True
)

[*********************100%***********************]  62 of 62 completed


In [23]:
data.head()

Ticker     WIPRO.NS                                               JPM  \
Price          Open     High      Low    Close        Volume     Open   
Date                                                                    
2024-04-15 218.6253 219.5158 214.7585 215.2975 12176472.0000 176.8782   
2024-04-16 212.3447 214.8991 208.4779 210.1418 21439424.0000 175.3442   
2024-04-17      NaN      NaN      NaN      NaN           NaN 173.8774   
2024-04-18 212.0869 213.4696 207.5639 208.2670 21986258.0000 173.6186   
2024-04-19 206.2281 212.7431 204.8220 212.2041 20470106.0000 174.8649   

Ticker                                               ...  SBIN.NS           \
Price          High      Low    Close        Volume  ...     Open     High   
Date                                                 ...                     
2024-04-15 179.7159 174.6732 175.3347 14766600.0000  ... 732.5591 735.9336   
2024-04-16 175.5935 172.2285 173.3310 16451800.0000  ... 724.3156 727.8348   
2024-04-17 174.8841 171.7971 172.6407  9017100.0000  ...      NaN      NaN   
2024-04-18 175.7853 172.5353 173.7624  9557700.0000  ... 725.0869 732.7037   
2024-04-19 178.2011 173.9158 178.1244 13402300.0000  ... 708.1662 725.0387   

Ticker                                     BAJAJFINSV.NS                      \
Price           Low    Close        Volume          Open      High       Low   
Date                                                                           
2024-04-15 721.9053 730.3416 11356572.0000     1670.1321 1690.1098 1652.3520   
2024-04-16 717.7112 724.7495 13338991.0000     1638.1678 1646.8580 1609.5497   
2024-04-17      NaN      NaN           NaN           NaN       NaN       NaN   
2024-04-18 715.3973 718.0969 14589648.0000     1623.1846 1628.2789 1587.5245   
2024-04-19 705.8040 723.5443 10886554.0000     1585.2270 1626.9304 1566.9474   

Ticker                             
Price          Close       Volume  
Date                               
2024-04-15 1654.9990  828843.0000  
2024-04-16 1616.6918 1528544.0000  
2024-04-17       NaN          NaN  
2024-04-18 1592.1194 1325537.0000  
2024-04-19 1617.1913 1242066.0000  

[5 rows x 310 columns]

In [24]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 518 entries, 2024-04-15 to 2026-04-13
Columns: 310 entries, ('WIPRO.NS', 'Open') to ('BAJAJFINSV.NS', 'Volume')
dtypes: float64(310)
memory usage: 1.2 MB


In [25]:
data.isnull().sum()

Ticker         Price 
WIPRO.NS       Open      24
               High      24
               Low       24
               Close     24
               Volume    24
                         ..
BAJAJFINSV.NS  Open      24
               High      24
               Low       24
               Close     24
               Volume    24
Length: 310, dtype: int64

In [26]:
data = data.swaplevel(axis=1).sort_index(axis=1)
raw=data
# 1. Ensure datetime index
raw.index.name = "Date"
raw.index = pd.to_datetime(raw.index, errors="coerce")

# 2. Drop invalid dates
raw = raw[raw.index.notna()].sort_index()

# 3. Convert all values to numeric
raw = raw.apply(pd.to_numeric, errors="coerce")

# 4. Handle missing values
raw = raw.ffill().bfill()

# 5. Drop completely empty columns
raw = raw.dropna(axis=1, how="all")

In [27]:
raw.head()

Price         Close                                                 \
Ticker         AAPL     ADBE     AMZN AXISBANK.NS       BA     BAC   
Date                                                                 
2024-04-15 171.1326 470.1000 183.6200   1056.2314 167.8200 34.3117   
2024-04-16 167.8524 476.2200 183.3200   1050.1914 170.5500 33.0996   
2024-04-17 166.4848 474.4500 181.2800   1050.1914 170.2100 33.6245   
2024-04-18 165.5335 473.1800 179.2200   1022.3366 170.2300 34.1399   
2024-04-19 163.5119 465.0200 174.6300   1027.4283 169.8200 35.2852   

Price                                                              ...  \
Ticker     BAJAJ-AUTO.NS BAJAJFINSV.NS BAJFINANCE.NS BRITANNIA.NS  ...   
Date                                                               ...   
2024-04-15     8703.9258     1654.9990      699.7724    4639.9087  ...   
2024-04-16     8628.5645     1616.6918      687.7078    4619.2520  ...   
2024-04-17     8628.5645     1616.6918      687.7078    4619.2520  ...   
2024-04-18     8727.3857     1592.1194      681.6706    4574.7227  ...   
2024-04-19     8517.6494     1617.1913      704.0494    4548.7065  ...   

Price            Volume                                            \
Ticker           TCS.NS     TECHM.NS       TMPV.NS           TSLA   
Date                                                                
2024-04-15 4200329.0000 2153147.0000 26953550.0000 100245300.0000   
2024-04-16 3051420.0000 2085459.0000 26953550.0000  97000000.0000   
2024-04-17 3051420.0000 2085459.0000 26953550.0000  82439700.0000   
2024-04-18 3476284.0000 3242357.0000 26953550.0000  96098800.0000   
2024-04-19 2957749.0000 2572144.0000 26953550.0000  87074500.0000   

Price                                                               \
Ticker               UNH             V           WFC      WIPRO.NS   
Date                                                                 
2024-04-15  5376800.0000 10267500.0000 19407500.0000 12176472.0000   
2024-04-16 11816500.0000  8237100.0000 25620000.0000 21439424.0000   
2024-04-17  8759800.0000  5940900.0000 18867700.0000 21439424.0000   
2024-04-18  8880400.0000  8231800.0000 24468500.0000 21986258.0000   
2024-04-19  6618600.0000  7905400.0000 34334300.0000 20470106.0000   

Price                                   
Ticker               WMT           XOM  
Date                                    
2024-04-15 10557200.0000 15029500.0000  
2024-04-16 14726300.0000 18082200.0000  
2024-04-17 15329700.0000 14538600.0000  
2024-04-18 12061500.0000 13821400.0000  
2024-04-19 14165800.0000 21572400.0000  

[5 rows x 310 columns]

In [28]:
def rolling_zscore(series, window=252):
    """Backward-looking rolling z-score — prevents look-ahead bias."""
    mu  = series.rolling(window, min_periods=63).mean()
    std = series.rolling(window, min_periods=63).std()
    return (series - mu) / (std + 1e-8)


def extract_ticker(df_wide, ticker):
    """Extract OHLCV for one ticker and add basic features."""
    stock = pd.DataFrame({
        "Open":   df_wide[("Open",   ticker)],
        "High":   df_wide[("High",   ticker)],
        "Low":    df_wide[("Low",    ticker)],
        "Close":  df_wide[("Close",  ticker)],
        "Volume": df_wide[("Volume", ticker)],
    })
    stock.index.name = "Date"
    stock = stock[~stock.index.duplicated(keep="first")]

    stock["return"]     = stock["Close"].pct_change()
    stock["log_return"] = np.log1p(stock["return"])   # equivalent, simpler
    stock = stock.dropna(subset=["return"])

    # ✅ Backward-looking rolling z-score — no look-ahead bias
    stock["vol_zscore"]  = rolling_zscore(stock["Volume"])
    stock["vol_outlier"] = (stock["vol_zscore"].abs() > CONFIG["outlier_sigma"]).astype(int)
    stock.drop(columns=["vol_zscore"], inplace=True)

    stock["Ticker"] = ticker
    return stock


frames  = [extract_ticker(raw, t) for t in tickers]
df_long = pd.concat(frames).sort_values(["Ticker", "Date"]).reset_index()
df_long.head()

,Date,Open,High,Low,Close,Volume,return,log_return,vol_outlier,Ticker
0,2024-04-16,170.2010,172.1929,166.7524,167.8524,73711200.0000,-0.0192,-0.0194,0,AAPL
1,2024-04-17,168.0803,169.1109,166.4848,166.4848,50901200.0000,-0.0081,-0.0082,0,AAPL
2,2024-04-18,166.5146,167.1191,165.0479,165.5335,43122900.0000,-0.0057,-0.0057,0,AAPL
3,2024-04-19,164.7110,164.8993,162.6002,163.5119,68149400.0000,-0.0122,-0.0123,0,AAPL
4,2024-04-22,164.0272,165.7515,163.2840,164.3443,48116400.0000,0.0051,0.0051,0,AAPL


In [29]:
df_long.isnull().sum()

Date           0
Open           0
High           0
Low            0
Close          0
Volume         0
return         0
log_return     0
vol_outlier    0
Ticker         0
dtype: int64

In [30]:
summary = []
for ticker, grp in df_long.groupby("Ticker"):
    summary.append({
        "Ticker":       ticker,
        "Rows":         len(grp),
        "Date From":    grp.Date.min().date(),
        "Date To":      grp.Date.max().date(),
        "Close Min":    round(grp.Close.min(), 2),
        "Close Max":    round(grp.Close.max(), 2),
        "Mean Ret%":    round(grp["return"].mean() * 100, 4),
        "Std Ret%":     round(grp["return"].std()  * 100, 4),
        "Skewness":     round(grp["return"].skew(), 3),
        "Kurtosis":     round(grp["return"].kurtosis(), 3),
        "Vol Outliers": int(grp["vol_outlier"].sum()),
    })

df_summary = pd.DataFrame(summary)
print("=" * 80)
print("STOCK SUMMARY STATISTICS")
print("=" * 80)
print(df_summary.to_string(index=False))

print()
print("📖 HOW TO READ THIS:")
print("  Skewness near 0  → symmetric returns")
print("  Kurtosis > 3     → fat tails (extreme moves happen more than expected)")
print("  High Vol Outliers → stock reacts strongly to news/earnings")
print()
print("📖 YOUR STOCKS:")
for _, row in df_summary.iterrows():
    note = "⚡ High kurtosis (fat tails)" if row['Kurtosis'] > 5 else "✅ Normal tail behavior"
    print(f"  {row['Ticker']:20s}: Skew={row['Skewness']:+.3f} | Kurt={row['Kurtosis']:.2f} | {note}")


STOCK SUMMARY STATISTICS
       Ticker  Rows  Date From    Date To  Close Min  Close Max  Mean Ret%  Std Ret%  Skewness  Kurtosis  Vol Outliers
         AAPL   517 2024-04-16 2026-04-13   163.5100   285.9200     0.0955    1.7505    0.9400   13.7130             5
         ADBE   517 2024-04-16 2026-04-13   225.3500   586.5500    -0.1078    2.0880   -0.7220   12.2300            11
         AMZN   517 2024-04-16 2026-04-13   161.0200   254.0000     0.0716    1.9999    0.1920    5.1370            14
  AXISBANK.NS   517 2024-04-16 2026-04-13   947.2900  1403.0000     0.0595    1.5184    0.0690    3.6860             5
           BA   517 2024-04-16 2026-04-13   136.5900   252.1500     0.0783    2.1970    0.3250    6.8820             8
          BAC   517 2024-04-16 2026-04-13    33.1000    56.9300     0.0980    1.5837   -0.6450    7.9110             7
BAJAJ-AUTO.NS   517 2024-04-16 2026-04-13  7126.1900 12353.3200     0.0360    1.5882   -0.9470    8.7220             7
BAJAJFINSV.NS   517 202